In [0]:
# Read the calendar
path = "/Volumes/workspace/default/m5"

calendar = spark.read.csv(f"{path}/calendar.csv", header=True, inferSchema=True)
calendar.printSchema()
calendar.show(5)

root
 |-- date: date (nullable = true)
 |-- wm_yr_wk: integer (nullable = true)
 |-- weekday: string (nullable = true)
 |-- wday: integer (nullable = true)
 |-- month: integer (nullable = true)
 |-- year: integer (nullable = true)
 |-- d: string (nullable = true)
 |-- event_name_1: string (nullable = true)
 |-- event_type_1: string (nullable = true)
 |-- event_name_2: string (nullable = true)
 |-- event_type_2: string (nullable = true)
 |-- snap_CA: integer (nullable = true)
 |-- snap_TX: integer (nullable = true)
 |-- snap_WI: integer (nullable = true)

+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+
|      date|wm_yr_wk|  weekday|wday|month|year|  d|event_name_1|event_type_1|event_name_2|event_type_2|snap_CA|snap_TX|snap_WI|
+----------+--------+---------+----+-----+----+---+------------+------------+------------+------------+-------+-------+-------+
|2011-01-29|   11101| Saturday|   1|    1|2011|d_1|    

In [0]:
# Read daily sales (one column per day)
sales = spark.read.csv(f"{path}/sales_train_evaluation.csv", header=True, inferSchema=True)
print("columns:", len(sales.columns))

columns: 1947


In [0]:
# Filter CA_1 / FOODS_3 and show that filter is lazy and count runs the job
import time

start = time.time()
ca1 = sales.filter((sales.store_id == "CA_1") & (sales.dept_id == "FOODS_3"))
print(f"filter took {time.time() - start:.2f} s")

start = time.time()
print("rows:", ca1.count())
print(f"count took {time.time() - start:.2f} s")

filter took 0.00 s
rows: 823
count took 0.65 s


In [0]:
# Wide to long, then join calendar and weekly prices
from pyspark.sql import functions as F

id_cols = ["id", "item_id", "dept_id", "cat_id", "store_id", "state_id"]
day_cols = [c for c in sales.columns if c.startswith("d_")]

#Wide to long (pandas melt)
long = ca1.unpivot(id_cols, day_cols, "d", "sales")

#Add real dates and events, keep the last 3 years (pandas merge + filter)
cal = calendar.select("d", "date", "wm_yr_wk", "event_name_1", "event_type_1", "snap_CA")
long = long.join(cal, on="d", how="left").filter(F.col("date") >= "2013-05-23")

#Add weekly prices
prices = spark.read.csv(f"{path}/sell_prices.csv", header=True, inferSchema=True)
long = long.join(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")

In [0]:
# Check against the pandas version 
long.agg(
    F.count("*").alias("rows"),
    F.sum("sales").alias("total_sales"),
    F.sum(F.col("sell_price").isNull().cast("int")).alias("rows_without_price"),
    F.round(F.sum("sell_price"), 2).alias("sum_price"),
    F.min("date").alias("min_date"),
    F.max("date").alias("max_date"),
).show()

+------+-----------+------------------+----------+----------+----------+
|  rows|total_sales|rows_without_price| sum_price|  min_date|  max_date|
+------+-----------+------------------+----------+----------+----------+
|902008|    2398726|             55370|2437795.33|2013-05-23|2016-05-22|
+------+-----------+------------------+----------+----------+----------+



In [0]:
#save as Delta table
long.write.mode("overwrite").saveAsTable("workspace.default.sales_long")
df = spark.table("workspace.default.sales_long")
print(df.count())

902008


In [0]:
# Rolling features with window functions
from pyspark.sql import Window 

# Off-shelf days are not real zero demand, so they become null 
df = df.withColumn("sales_on_shelf", F.when(F.col("sell_price").isNotNull(), F.col("sales")))

def last_n_days(n):
    #For each item, ordered by date
    return Window.partitionBy("id").orderBy("date").rowsBetween(-(n - 1), 0)

features = (
    df.withColumn("mean_7", F.avg("sales_on_shelf").over(last_n_days(7)))
      .withColumn("mean_28", F.avg("sales_on_shelf").over(last_n_days(28)))
      .withColumn("mean_112", F.avg("sales_on_shelf").over(last_n_days(112)))
      .withColumn("zero_share_28", F.avg(
          F.when(F.col("sales_on_shelf") == 0, 1).when(F.col("sales_on_shelf").isNotNull(), 0)
      ).over(last_n_days(28)))
)

features.filter(F.col("date") == "2016-04-24").agg(
    F.count("*").alias("items"),
    F.round(F.sum("mean_7"), 4).alias("sum_mean_7"),
    F.round(F.sum("mean_28"), 4).alias("sum_mean_28"),
    F.round(F.sum("mean_112"), 4).alias("sum_mean_112"),
    F.round(F.sum("zero_share_28"), 4).alias("sum_zero_share_28"),
).show()

+-----+----------+-----------+------------+-----------------+
|items|sum_mean_7|sum_mean_28|sum_mean_112|sum_zero_share_28|
+-----+----------+-----------+------------+-----------------+
|  823| 2191.7143|  2201.8571|   2071.2346|         323.0357|
+-----+----------+-----------+------------+-----------------+



In [0]:
# Prepare sales, calendar and prices for all 10 stores and save as a Delta table
import time
start = time.time()

all_long = (
    sales.unpivot(id_cols, day_cols, "d", "sales")
    .join(calendar.select("d", "date", "wm_yr_wk", "event_name_1", "event_type_1",
                          "snap_CA", "snap_TX", "snap_WI"), on="d", how="left")
    .filter(F.col("date") >= "2013-05-23")
    .join(prices, on=["store_id", "item_id", "wm_yr_wk"], how="left")
    # Each store uses the SNAP days of its own state
    .withColumn("snap", F.when(F.col("state_id") == "CA", F.col("snap_CA"))
                         .when(F.col("state_id") == "TX", F.col("snap_TX"))
                         .otherwise(F.col("snap_WI")))
    .drop("snap_CA", "snap_TX", "snap_WI")
)
all_long.write.mode("overwrite").saveAsTable("workspace.default.sales_long_all")
print(f"prepared in {time.time() - start:.0f} s")

prepared in 10 s


In [0]:
# Add rolling features for all stores, save them and check series per store
def add_features(df):
    df = df.withColumn("sales_on_shelf", F.when(F.col("sell_price").isNotNull(), F.col("sales")))
    return (
        df.withColumn("mean_7", F.avg("sales_on_shelf").over(last_n_days(7)))
          .withColumn("mean_28", F.avg("sales_on_shelf").over(last_n_days(28)))
          .withColumn("mean_112", F.avg("sales_on_shelf").over(last_n_days(112)))
          .withColumn("zero_share_28", F.avg(
              F.when(F.col("sales_on_shelf") == 0, 1).when(F.col("sales_on_shelf").isNotNull(), 0)
          ).over(last_n_days(28)))
    )

start = time.time()
all_features = add_features(spark.table("workspace.default.sales_long_all"))
all_features.write.mode("overwrite").saveAsTable("workspace.default.features_all")
print(f"features in {time.time() - start:.0f} s")

spark.table("workspace.default.features_all").groupBy("store_id").agg(
    F.countDistinct("id").alias("series"),
    F.count("*").alias("rows"),
    F.sum("sales").alias("total_sales"),
).orderBy("store_id").show()

features in 14 s
+--------+------+-------+-----------+
|store_id|series|   rows|total_sales|
+--------+------+-------+-----------+
|    CA_1|  3049|3341704|    4813489|
|    CA_2|  3049|3341704|    3547354|
|    CA_3|  3049|3341704|    6924166|
|    CA_4|  3049|3341704|    2615665|
|    TX_1|  3049|3341704|    3408565|
|    TX_2|  3049|3341704|    4223880|
|    TX_3|  3049|3341704|    3843051|
|    WI_1|  3049|3341704|    3611518|
|    WI_2|  3049|3341704|    4564845|
|    WI_3|  3049|3341704|    3508637|
+--------+------+-------+-----------+

